I will construct GLM from exponential distribution and use Newton's method to update \theta
\begin{align*}
p(y;\eta)=b(y)\exp(\eta^T T(y)-a(\eta))\\
p(y;\lambda) = \lambda e^{(-\lambda y)}\\
p(y;\lambda) = e^{(\log(\lambda) - \lambda y)}\\
\eta=-\lambda,a(\eta)=-\log\lambda,b(y)=1,\lambda=-\eta\\
E[T(y);\eta]=\frac{1}{\lambda}=\frac{-1}{\eta}=\frac{-1}{\theta^{T}x}\\
l(\theta)=\sum_{i=1}^{m}\log e^{(\log(\lambda^{(i)}) - \lambda^{(i)} y^{(i)})}=\sum_{i=1}^{m}\eta^{(i)} y^{(i)}+\log-\eta^{(i)}=\\
=\sum_{i=1}^{m}\theta^{T}x^{(i)}y^{(i)}+\log-\theta^{T}x^{(i)}\\
\nabla_\theta l(\theta)=\sum_{i=1}^{m}(y^{(i)}+\frac{1}{\theta^{T}x^{(i)}})x^{(i)}=X^{T}(y+\frac{1}{X\theta})\\
H=\nabla_{\theta}^{2} l(\theta)=-\sum_{i=1}^{m}\frac{1}{(\theta^{T}x^{(i)})^{2}}x^{(i)}(x^{(i)})^{T}=-X^{T}WX\\
W=diag(\frac{1}{(X\theta)^{2}}),W_{ii}=\frac{1}{(\theta^{T}x^{(i)})^{2}}\\
\theta:=\theta-H^{-1}\nabla_\theta l(\theta)
\end{align*}

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
class GLM:
    def __init__(self,eps=1e-5,max_iter=100):
        self.eps=eps
        self.max_iter=max_iter
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        m,n=self.X.shape
        self.theta=np.zeros(n)-0.1
        prev_theta=self.theta+2*self.eps
        it=0
        while np.linalg.norm(self.theta-prev_theta,ord=1)>=self.eps and it<self.max_iter:
            it+=1
            prev_theta=self.theta.copy()
            X_theta=self.X@self.theta
            
            gradient=self.X.T@(self.y+1/(X_theta))
            W=np.diag(1/(X_theta)**2)
            hessian=-X.T@W@X
            self.theta=self.theta-np.linalg.inv(hessian)@gradient
        print(f"Converged in {it} iterations")
    def predict(self,X):
        X=np.asarray(X)
        return -1/(X@self.theta)

In [3]:
random_seed=122
m,n=10000,3
rng = np.random.RandomState(random_seed)
X = rng.uniform(0.1, 5.0, (m, n))
X[:, 0] = 1.0
original_theta=np.array([-0.3,-1.1,-0.45])
comb=X@original_theta
expected=-1/comb
y_correct=rng.exponential(scale=expected)
n_swap=0.3
indexs=rng.choice(range(m),size=int(m*n_swap),replace=False)
y_mixed=y_correct.copy()             
y_mixed[indexs]=rng.normal(loc=5, 
    scale=3)
y_mixed = np.clip(y_mixed, a_min=1e-5, a_max=None)
y=np.vstack([y_correct,y_mixed]).T

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [5]:
y_train_correct,y_train_mixed=y_train[:,0],y_train[:,1]
y_test_correct,y_test_mixed=y_test[:,0],y_test[:,1]

In [6]:
glm=GLM()
glm.fit(X_train,y_train_correct)
preds_correct=glm.predict(X_test)
print(f"Average error for correct test set: {np.round(np.mean(np.abs(y_test_correct-preds_correct)),2)}")


Converged in 8 iterations
Average error for correct test set: 0.21


In [7]:
glm.theta

array([-0.337147  , -1.11970102, -0.42077344])

In [8]:
glm=GLM()
glm.fit(X_train,y_train_mixed)
preds_mixed=glm.predict(X_test)
print(f"Average error for mixed test set: {np.round(np.mean(np.abs(y_test_mixed-preds_mixed)),2)}")

Converged in 5 iterations
Average error for mixed test set: 2.29


In [9]:
glm.theta

array([-0.4766171 , -0.01329099, -0.00299022])